# Exercise 09 — Fuse multiple NYC graphs (answer)

Every model receives the same local temporal features: demand at $t-1$, $t-2$, $t-3$, $t-24$, and $t-168$, plus calendar variables. The local ANN processes each taxi zone independently. A GCN adds neighbour summaries defined by one or more graphs.

To protect the strong signal from a zone's own recent demand, the graph models keep a separate self-node branch and predict a correction to persistence. We select graph combinations on validation data, refit the selected designs on train + validation, and use the test set once for the final comparison.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `09 Graph neural networks`, so relative data paths work. The answer repository is private, so this setup deliberately clones the public `AI-Code` repository; it contains the same lesson data files.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "09 Graph neural networks"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics. PyTorch builds the neural networks used in this lesson.


In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("torch", torch.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn torch


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.linear_model import LinearRegression, Ridge

plt.style.use("seaborn-v0_8-whitegrid")
torch.manual_seed(42)
np.random.seed(42)

flow = pd.read_csv("nyc_zone_hour.csv.gz", parse_dates=["time"])
zones = pd.read_csv("nyc_zone_features.csv").sort_values("LocationID")
connectivity = pd.read_csv("nyc_training_connectivity.csv.gz")
geo = json.load(open("nyc_taxi_zones.geojson"))

zone_ids = zones["LocationID"].astype(int).tolist()
zone_names = zones["Zone"].tolist()
n_zones = len(zone_ids)
zone_position = {zone_id: i for i, zone_id in enumerate(zone_ids)}
demand = flow.pivot(index="time", columns="zone_id", values="pickups")
demand = demand.reindex(columns=zone_ids)

def haversine_matrix(latitude, longitude):
    lat_i = np.radians(latitude[:, None])
    lat_j = np.radians(latitude[None, :])
    delta_lon = np.radians(longitude[:, None] - longitude[None, :])
    value = (
        np.sin((lat_i - lat_j) / 2) ** 2
        + np.cos(lat_i) * np.cos(lat_j) * np.sin(delta_lon / 2) ** 2
    )
    return 6371 * 2 * np.arcsin(np.sqrt(value))

latitude = zones["latitude"].to_numpy()
longitude = zones["longitude"].to_numpy()
distance = haversine_matrix(latitude, longitude)
distance_scale = np.median(distance[distance > 0])

poi_columns = [
    "food_facilities",
    "health_facilities",
    "retail_facilities",
    "education_facilities",
    "transport_facilities",
]
poi = np.log1p(zones[poi_columns].to_numpy())

mobility = np.zeros((n_zones, n_zones))
for row in connectivity.itertuples():
    if row.origin in zone_position and row.destination in zone_position:
        i = zone_position[row.origin]
        j = zone_position[row.destination]
        mobility[i, j] += row.flow
        mobility[j, i] += row.flow

base_graphs = {
    "Geographical": np.exp(-(distance / distance_scale) ** 2),
    "POI": cosine_similarity(poi),
    "Mobility": np.log1p(mobility),
}

def neighbour_matrix(weights, k=8):
    weights = weights.copy()
    np.fill_diagonal(weights, 0)
    retained = np.zeros_like(weights)
    for i in range(n_zones):
        neighbours = np.argpartition(weights[i], -k)[-k:]
        retained[i, neighbours] = weights[i, neighbours]
    retained = np.maximum(retained, retained.T)
    np.fill_diagonal(retained, 0)
    row_sum = retained.sum(axis=1, keepdims=True)
    return np.divide(
        retained, row_sum, out=np.zeros_like(retained), where=row_sum > 0
    )

graphs = {name: neighbour_matrix(weights, k=8) for name, weights in base_graphs.items()}

def polygons(feature):
    geometry = feature["geometry"]
    return (
        geometry["coordinates"]
        if geometry["type"] == "MultiPolygon"
        else [geometry["coordinates"]]
    )

def zone_map(values, title, cmap="viridis", ax=None, colorbar_label=None):
    values = pd.Series(values)
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 7))
    upper = values.max() if values.max() > values.min() else values.min() + 1
    norm = plt.Normalize(values.min(), upper)
    for feature in geo["features"]:
        zone_id = int(feature["properties"]["LocationID"])
        value = values.get(zone_id, np.nan)
        facecolor = "lightgrey" if pd.isna(value) else plt.colormaps[cmap](norm(value))
        for polygon in polygons(feature):
            ax.fill(*np.asarray(polygon[0]).T, facecolor=facecolor, edgecolor="white", linewidth=.45)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.axis("off")
    colorbar = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax, shrink=.72)
    if colorbar_label:
        colorbar.set_label(colorbar_label)
    return ax

values = demand.to_numpy(float)
times = demand.index
lags = [1, 2, 3, 24, 168]
feature_names = [
    "lag_1", "lag_2", "lag_3", "lag_24", "lag_168",
    "hour_sin", "hour_cos", "weekend",
]
samples = []
targets = []
target_times = []
for t in range(max(lags), len(values)):
    calendar = [
        np.sin(2 * np.pi * times[t].hour / 24),
        np.cos(2 * np.pi * times[t].hour / 24),
        int(times[t].dayofweek >= 5),
    ]
    local_history = [values[t - lag] for lag in lags]
    calendar_columns = [np.full(n_zones, item) for item in calendar]
    samples.append(np.column_stack(local_history + calendar_columns))
    targets.append(values[t])
    target_times.append(times[t])

X_raw = np.asarray(samples)
y = np.asarray(targets)
target_times = pd.DatetimeIndex(target_times)
train_mask = target_times < "2024-02-10"
valid_mask = (target_times >= "2024-02-10") & (target_times < "2024-02-20")
test_mask = target_times >= "2024-02-20"

train_mean = y[train_mask].mean(axis=0)
train_std = y[train_mask].std(axis=0).clip(1)

# Raw arrays are kept for student inspection and final reporting.
X_train = X_raw[train_mask]
y_train = y[train_mask]
X_valid = X_raw[valid_mask]
y_valid = y[valid_mask]
X_test = X_raw[test_mask]
y_test = y[test_mask]

# Models use a separate standardized copy.
X = X_raw.copy()
X[:, :, :5] = (X[:, :, :5] - train_mean[None, :, None]) / train_std[None, :, None]
y_standardized = (y - train_mean) / train_std
X_train_model = X[train_mask]
y_train_model = y_standardized[train_mask]
X_valid_model = X[valid_mask]
y_valid_model = y_standardized[valid_mask]

print("Raw supervised-learning arrays:")
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print("X = samples × taxi-zone nodes × local temporal features")
print("y = samples × taxi-zone nodes; values are pickups in the next hour")
print("Feature order:", feature_names)
print("Example raw feature row for", zone_names[0], "(lag values are pickups/hour):")
display(pd.DataFrame([X_train[0, 0]], columns=feature_names))
print("Scaling is used only inside model fitting; all reported errors use pickups/hour.")

## Question 1 — Read and compare the graph definitions

For each adjacency matrix, visualise the row-normalised weights and summarise sparsity. Explain the message-passing operation

$$H_i=\sum_j A_{ij}X_j$$

and why geographical, POI and mobility neighbours need not coincide.

In [ ]:
# TODO: visualise each adjacency matrix and summarise its non-zero weights.
# TODO: explain H_i = sum_j A_ij X_j in words.

## Prepared model classes

`LocalANN` uses only node-local features. A single-graph GCN adds one neighbour representation; the multi-graph model learns a separate gated representation for each adjacency matrix.

In [ ]:
class LocalANN(nn.Module):
    def __init__(self, n_features, hidden=32):
        super().__init__()
        self.hidden = nn.Linear(n_features, hidden)
        self.output = nn.Linear(hidden, 1)

    def forward(self, x):
        # x is already (samples × zones, features).
        local_hidden = torch.relu(self.hidden(x))
        correction = self.output(local_hidden).squeeze(-1)
        return x[:, 0] + correction


class SingleGraphGCN(nn.Module):
    def __init__(self, graph, n_features, hidden=32):
        super().__init__()
        self.register_buffer("graph", torch.tensor(graph, dtype=torch.float32))
        self.self_layer = nn.Linear(n_features, hidden)
        self.neighbour_layer = nn.Linear(n_features, hidden, bias=False)
        self.neighbour_gate = nn.Parameter(torch.tensor(-2.0))
        self.output = nn.Linear(2 * hidden, 1)

    def forward(self, x):
        neighbour_x = torch.einsum("ij,bjf->bif", self.graph, x)
        local_hidden = torch.relu(self.self_layer(x))
        neighbour_hidden = torch.relu(self.neighbour_layer(neighbour_x))
        gate = torch.sigmoid(self.neighbour_gate)
        combined = torch.cat([local_hidden, gate * neighbour_hidden], dim=-1)
        correction = self.output(combined).squeeze(-1)
        return x[:, :, 0] + correction


class MultiGraphGCN(nn.Module):
    def __init__(self, graph_list, n_features, hidden=32):
        super().__init__()
        for graph_number, graph in enumerate(graph_list):
            self.register_buffer(
                f"graph_{graph_number}", torch.tensor(graph, dtype=torch.float32)
            )
        self.n_graphs = len(graph_list)
        self.self_layer = nn.Linear(n_features, hidden)
        self.neighbour_layers = nn.ModuleList(
            [nn.Linear(n_features, hidden, bias=False) for _ in graph_list]
        )
        self.neighbour_gates = nn.Parameter(torch.full((self.n_graphs,), -2.0))
        self.output = nn.Linear(hidden * (1 + self.n_graphs), 1)

    def forward(self, x):
        representations = [torch.relu(self.self_layer(x))]
        gates = torch.sigmoid(self.neighbour_gates)
        for graph_number, layer in enumerate(self.neighbour_layers):
            graph = getattr(self, f"graph_{graph_number}")
            neighbour_x = torch.einsum("ij,bjf->bif", graph, x)
            neighbour_hidden = torch.relu(layer(neighbour_x))
            representations.append(gates[graph_number] * neighbour_hidden)
        combined = torch.cat(representations, dim=-1)
        correction = self.output(combined).squeeze(-1)
        return x[:, :, 0] + correction


def neural_arrays(model, mask):
    if isinstance(model, LocalANN):
        x_model = X[mask].reshape(-1, X.shape[-1])
        y_model = y_standardized[mask].reshape(-1)
    else:
        x_model = X[mask]
        y_model = y_standardized[mask]
    return x_model, y_model


def fit_with_validation(model, max_epochs=500, patience=40, min_delta=5e-4):
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    loss_function = nn.MSELoss()
    x_train_array, y_train_array = neural_arrays(model, train_mask)
    x_valid_array, y_valid_array = neural_arrays(model, valid_mask)
    x_train_tensor = torch.tensor(x_train_array, dtype=torch.float32)
    y_train_tensor = torch.tensor(y_train_array, dtype=torch.float32)
    x_valid_tensor = torch.tensor(x_valid_array, dtype=torch.float32)
    y_valid_tensor = torch.tensor(y_valid_array, dtype=torch.float32)
    best_loss = float("inf")
    best_state = None
    best_epoch = 0
    waiting = 0
    for epoch in range(max_epochs):
        model.train()
        optimizer.zero_grad()
        train_loss = loss_function(model(x_train_tensor), y_train_tensor)
        train_loss.backward()
        optimizer.step()
        model.eval()
        with torch.no_grad():
            valid_loss = loss_function(model(x_valid_tensor), y_valid_tensor).item()
        if valid_loss < best_loss - min_delta:
            best_loss = valid_loss
            best_state = {
                key: value.detach().clone() for key, value in model.state_dict().items()
            }
            best_epoch = epoch + 1
            waiting = 0
        else:
            waiting += 1
        if waiting >= patience:
            break
    model.load_state_dict(best_state)
    return model, best_epoch


def fit_fixed_epochs(model, epochs):
    fit_mask = train_mask | valid_mask
    x_fit_array, y_fit_array = neural_arrays(model, fit_mask)
    x_fit = torch.tensor(x_fit_array, dtype=torch.float32)
    y_fit = torch.tensor(y_fit_array, dtype=torch.float32)
    optimizer = torch.optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-4)
    loss_function = nn.MSELoss()
    for _ in range(max(1, int(epochs))):
        model.train()
        optimizer.zero_grad()
        loss = loss_function(model(x_fit), y_fit)
        loss.backward()
        optimizer.step()
    return model


def predict_demand(model, mask):
    x_model, _ = neural_arrays(model, mask)
    model.eval()
    with torch.no_grad():
        prediction = model(torch.tensor(x_model, dtype=torch.float32)).numpy()
    prediction = np.asarray(prediction).reshape(mask.sum(), len(train_mean))
    return np.clip(prediction * train_std + train_mean, 0, None)

## Question 2 — Tune local baselines and single-graph GCNs

Compare Linear, Ridge with α ∈ {0.1, 1, 10}, a local ANN, and one GCN per graph on the validation period. The tabular models use the same local lags and calendar features but cannot aggregate neighbouring zones.

In [ ]:
# TODO: tune Linear/Ridge and fit the local ANN and three single-graph GCNs.
# TODO: freeze settings using validation MAE only.

## Question 3 — Extension: fuse graphs

Compare geographical + POI, geographical + mobility, and all three graphs. Freeze the best single graph and best graph combination using validation MAE. Explain how the learned graph gates differ from simply averaging adjacency matrices.

In [ ]:
# TODO: train the three graph combinations and freeze the validation winner.
# TODO: explain the role of the learned graph gates.

## Question 4 — Refit, test once and diagnose errors

Refit all selected model families on train + validation for their selected epoch/setting. Compare test MAE/RMSE once, then visualise (i) where multi-graph fusion helps relative to the best single graph, (ii) zone-level test MAE as a filled map of all 67 Manhattan taxi-zone polygons, and (iii) MAE by hour.

In [ ]:
# TODO: refit selected families on train + validation and evaluate test once.
# TODO: plot fusion gain by zone, test MAE on the map, and MAE by hour.